In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [2]:
# Max Sharpe Ratio, Normal Assumption, w > 0
def negative_sharpe_ratio(w: np.ndarray, mu: np.ndarray, cov: np.ndarray, rf: float) -> float:
    port_return = w @ mu
    port_vol = np.sqrt(w.T @ cov @ w)
    sharpe = (port_return - rf) / port_vol
    return -sharpe

# Load covariance matrix and expected returns
cov_input = pd.read_csv("/Users/fuyuxuan/Downloads/test5_2.csv")
mean_input= pd.read_csv("/Users/fuyuxuan/Downloads/test10_3_means.csv")

# Convert to numpy array
cov_matrix = cov_input.values
mu = mean_input.iloc[:, 0].values

rf = 0.04

# Number of assets
n = len(mu)

# Initial guess: equal weights
w0 = np.ones(n) / n

# Constraint: weights sum to 1
constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

# Constraint: w > 0
for i in range(n):
    constraints.append({"type": "ineq", "fun": lambda w, i=i: w[i]})
bounds = [(0, 1) for _ in range(n)]

# Optimize
result = minimize(
    negative_sharpe_ratio,
    w0,
    args=(mu, cov_matrix, rf),
    method="SLSQP",
    constraints=constraints,
    options={"ftol": 1e-15, "maxiter": 1000}
)

# Get optimized weights
weights = result.x

output = pd.DataFrame(weights, columns=["W"])
print(output)

              W
0 -2.355675e-15
1 -2.383510e-14
2 -1.484107e-16
3  1.213108e-01
4  8.786892e-01
